# 02b — FP32 re-prediction of the 45 main runs

The original runs made their final val/test predictions under fp16 autocast, so the saved logits are coarse
(~8% exact ties per test set). This notebook loads each run's `best.pt` and predicts val and test again in
full FP32. **No retraining:** same weights, same images, only the precision of the forward pass changes.
The result is the FP32 reference for the Core ML compression study.

**Setup:** dataset `paultimothymooney/chest-xray-pneumonia` **plus the 5 training outputs** (the saved versions of
the 5 `02_train` copies) as inputs, Internet **on**, accelerator **GPU T4 ×1**.

**How to run:** Run All interactively (or Save & Run All). Cell 4 re-predicts one run first and stops if the
check fails; cell 5 does the rest (~20–30 min). Download the output as `02b_repredict_output/` into the repo root.

In [ ]:
!pip install -q --no-deps --force-reinstall --no-cache-dir git+https://github.com/meap11/ppneu.git
try:
    import timm
except ImportError:
    !pip install -q timm

In [ ]:
import os, shutil, json
from pathlib import Path
import pandas as pd, torch
from ppneu import audit as A, data as D, train as T

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Turn on a GPU accelerator"
OUT = "/kaggle/working/runs"

root, _ = A.find_root("/kaggle/input")
manifests = {s: D.load_manifest(s) for s in ["official", "image_random", "grouped"]}
cache = D.build_cache(manifests["official"], root, "/kaggle/working/cache/cache256.npy", workers=os.cpu_count())
print("cache", cache.shape)

In [ ]:
# Find every finished training run in the attached outputs (mounted at varying depths)
runs = {}
for pat in ("*/runs", "*/*/runs", "*/*/*/runs", "*/*/*/*/runs"):
    for r in Path("/kaggle/input").glob(pat):
        for best in r.glob("*/*/seed*/best.pt"):
            key = best.parent.relative_to(r).parts            # (split, arch, seedK)
            if key in runs:
                raise ValueError(f"run {key} found twice: {runs[key]} and {best.parent}")
            runs[key] = best.parent
print(len(runs), "runs found")
print(pd.Series([k[1] for k in runs]).value_counts().to_dict())
assert len(runs) == 45, "expected 45 runs: attach all 5 training outputs"


In [ ]:
# Check one run first: FP32 logits should be close to the fp16 ones (median difference), AUROC nearly unchanged, ties ~0
key = sorted(runs)[0]
cmp = T.repredict_fp32(runs[key], manifests[key[0]], cache, OUT)
print(key, json.dumps(cmp, indent=1))
assert cmp["median_abs_logit_diff"] < 0.05, "logits moved too much: weights or preprocessing do not match"
assert abs(cmp["test_auc_new"] - cmp["test_auc_old"]) < 0.002
assert cmp["test_ties_new"] < 0.01

In [ ]:
rows = []
for i, key in enumerate(sorted(runs), 1):
    cmp = T.repredict_fp32(runs[key], manifests[key[0]], cache, OUT)
    rows.append({"split": key[0], "arch": key[1], "seed": key[2], **cmp})
    print(f"{i:2d}/{len(runs)} {'/'.join(key):40s} max|dz| {cmp['max_abs_logit_diff']:.4f} "
          f"AUROC {cmp['test_auc_old']:.4f} -> {cmp['test_auc_new']:.4f} ties {cmp['test_ties_new']:.4f}")
report = pd.DataFrame(rows)
report.to_csv(Path(OUT) / "repredict_report.csv", index=False)
display(report.describe().round(5))

In [ ]:
# Free space: the image cache is not needed in the saved output
shutil.rmtree("/kaggle/working/cache", ignore_errors=True)
!du -sh /kaggle/working/*